# 2D brightfield segmentation with nnU-Net v2

- Two independent 2D nnU-Net models trained from single-channel brightfield `.tiff` images with matching (binarised) label masks: one for **organoids**, one for **rosettes**.
- Each image/label pair is converted to NIfTI before training. Pixel spacing is read from each image's TIFF header; labels have no spacing metadata of their own, so they reuse the spacing of their paired image.
- Images (for both training and prediction) may have varying pixel sizes -> nnU-Net resamples to a consistent physical spacing during preprocessing/inference, provided the true spacing is embedded in every case.
- Note! This is a semantic segmentation -> touching objects are merged and may require watershed splitting.

In [1]:
import os, json, csv, subprocess, math
from pathlib import Path
import numpy as np
import tifffile
import SimpleITK as sitk
import torch
import matplotlib.pyplot as plt
from nnunetv2.dataset_conversion.generate_dataset_json import generate_dataset_json

if torch.cuda.is_available():
      print("Using GPU:", torch.cuda.get_device_name(0))

Using GPU: NVIDIA GeForce RTX 5080


In [2]:
def _raw_spacing_xy(img_path):
    """(x, y) pixel spacing read directly from the TIFF header, with no validation.
    Either component is None if its resolution tag is absent."""
    with tifffile.TiffFile(str(img_path)) as tif:
        tags = {t.name: t.value for t in tif.pages[0].tags.values()}
        xr = tags.get("XResolution")
        yr = tags.get("YResolution")
    x_um = xr[1] / xr[0] if xr and xr[0] else None  # XResolution = pixels per unit -> unit per pixel
    y_um = yr[1] / yr[0] if yr and yr[0] else None
    return x_um, y_um


def _is_bogus_spacing(val):
    """True if a spacing value is absent, or is the ~1.0 placeholder some tools write when no real
    calibration was ever set (i.e. not a genuine pixel size)."""
    return val is None or math.isclose(val, 1.0, abs_tol=1e-6)


def get_spacing_xy(img_path, override_cm=None):
    """Per-image (x, y) pixel spacing read from the TIFF header.
    Labels carry no resolution tags of their own -> callers should reuse the spacing of their paired image.
    A missing tag or a ~1.0 placeholder value counts as missing spacing.
    override_cm: used instead of raising when spacing is missing/placeholder."""
    x_um, y_um = _raw_spacing_xy(img_path)
    if _is_bogus_spacing(x_um) or _is_bogus_spacing(y_um):
        if override_cm is not None:
            return (override_cm, override_cm)
        raise ValueError(f'{Path(img_path).name}: missing/placeholder XResolution/YResolution pixel-size tag')
    return (x_um, y_um)


def check_pixel_metadata(img_paths, override_cm=None):
    """Flag any images with a missing or ~1.0-placeholder XResolution/YResolution pixel-size tag, so they
    can be fixed or removed before the main conversion/prediction pipeline runs. If override_um is given,
    offending images are still listed but noted as using the override instead of needing a fix.
    Returns the list of offending paths."""

    missing_paths = [path for path in img_paths if any(_is_bogus_spacing(v) for v in _raw_spacing_xy(path))]
    if missing_paths:
        if override_cm is not None:
            print(f"{len(missing_paths)}/{len(img_paths)} images missing pixel-size metadata -> "
                  f"using override_cm={override_cm} for these")
        else:
            print("Missing some pixel metadata! Pls fix before continuing!!")
        for path in missing_paths:
            print(path)
    else:
        print(f"All {len(img_paths)} images have pixel-size metadata")
    return missing_paths

In [3]:
def load_pair(img_path, labels_dir, label_ext):
    """
    Load brightfield (always with attached metadata) and matching label (often missing metadata) -> load images and their (shared) pixel spacing.
    """
    bf = np.asarray(tifffile.imread(str(img_path))).astype(np.float32)
    mask_path = labels_dir / (img_path.stem + label_ext)
    mask = ((np.asarray(tifffile.imread(str(mask_path))))>0).astype(np.uint8)
    if bf.shape != mask.shape:
        raise ValueError(f"Image and matching label don't have the same shape! Image:{img_path.name}, BF shape: {bf.shape}, Label shape: {mask.shape}")
    spacing_xy = get_spacing_xy(img_path)
    return bf, mask, spacing_xy


def write_nii_2d(arr_yx, path, is_label, spacing_xy):
    """Write a 2D array as a (1,Y,X) NIfTI. 
    NIfTI format is inherintely 3d, so we make a fake z axis (huge spacing)
    """
    img = sitk.GetImageFromArray(arr_yx[None].astype(np.uint8 if is_label else np.float32))
    img.SetSpacing((float(spacing_xy[0]), float(spacing_xy[1]), 999.0))   # sitk order (x,y,z)
    sitk.WriteImage(img, str(path))

In [4]:
def build_dataset(img_paths, labels_dir, label_ext, raw_ds):
    """Convert every matched (image, label) pair into nnU-Net raw imagesTr/labelsTr NIfTIs.
    Writes a provenance.csv alongside raw_ds and returns the per-case rows."""
    (raw_ds / 'imagesTr').mkdir(parents=True, exist_ok=True)
    (raw_ds / 'labelsTr').mkdir(parents=True, exist_ok=True)
    rows = []
    spacings = []
    for k, ip in enumerate(img_paths):
        case = f'case_{k:03d}'
        bf, mask, spacing_xy = load_pair(ip, labels_dir, label_ext)
        write_nii_2d(bf,   raw_ds / 'imagesTr' / f'{case}_0000.nii.gz', is_label=False, spacing_xy=spacing_xy)
        write_nii_2d(mask, raw_ds / 'labelsTr' / f'{case}.nii.gz',      is_label=True,  spacing_xy=spacing_xy)
        frac = 100.0 * float(mask.mean())
        spacings.append(spacing_xy)
        rows.append((case, ip.stem, round(frac, 2), round(spacing_xy[0], 4), round(spacing_xy[1], 4)))
        print(f'  {case}  {ip.stem[:44]:44s}  foreground={frac:5.2f}%  spacing_xy={spacing_xy}')
    with open(raw_ds / 'provenance.csv', 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['case', 'source_stem', 'foreground_pct', 'spacing_x_um', 'spacing_y_um'])
        w.writerows(rows)
    xs = sorted(s[0] for s in spacings)
    if xs and (xs[-1] - xs[0]) > 1e-6:
        print(f'  ! spacing varies across this dataset: x_um {xs[0]:.4f} - {xs[-1]:.4f} '
              '(fine for nnU-Net, which resamples to a target spacing during preprocessing)')
    return rows

In [5]:
def setup_nnunet_dirs(nnunet_base):
    """Create/point nnU-Net at raw/preprocessed/results folders under nnunet_base and set its env vars.
    Returns a dict with the three paths."""
    nnunet_raw          = nnunet_base / 'nnUNet_raw'
    nnunet_preprocessed = nnunet_base / 'nnUNet_preprocessed'
    nnunet_results      = nnunet_base / 'nnUNet_results'
    for d in (nnunet_raw, nnunet_preprocessed, nnunet_results):
        d.mkdir(parents=True, exist_ok=True)
    os.environ['nnUNet_raw']          = str(nnunet_raw)
    os.environ['nnUNet_preprocessed'] = str(nnunet_preprocessed)
    os.environ['nnUNet_results']      = str(nnunet_results)
    os.environ.pop('nnUNet_n_proc_DA', None)   # clear stale value
    return {'raw': nnunet_raw, 'preprocessed': nnunet_preprocessed, 'results': nnunet_results}


def run(cmd, da_workers=None):
    """Stream an nnU-Net CLI command. da_workers=0 -> single-threaded augmentation (Windows)."""
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    if da_workers is not None:
        env['nnUNet_n_proc_DA'] = str(da_workers)
        env.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
        env['OMP_NUM_THREADS'] = '1'
        env['MKL_NUM_THREADS'] = '1'
    print('>', cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, env=env, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

In [6]:
def read_timelapse_bf(path):
    """(Z, Y, X) float32 brightfield from a stacked TIFF; channel 0 if a channel axis exists."""
    arr = np.asarray(tifffile.imread(str(path))).astype(np.float32)
    if arr.ndim == 2:
        arr = arr[None]
    elif arr.ndim == 4:                                   # (Z, C, Y, X) -> channel 0
        arr = np.take(arr, 0, axis=1 if arr.shape[1] <= 4 else 0)
    return arr


def stage_stack(stack_path, stage_dir, index, override_cm=None):
    """Write every slice of one time-lapse stack as an independent 2D NIfTI case.
    Returns (spacing_xy, n_slices)."""
    bf = read_timelapse_bf(stack_path)
    spacing_xy = get_spacing_xy(stack_path, override_cm=override_cm)   # one spacing per stack, shared by all its slices
    for z in range(bf.shape[0]):
        write_nii_2d(bf[z], stage_dir / f'{index:03d}_{z:04d}_0000.nii.gz', is_label=False, spacing_xy=spacing_xy)
    return spacing_xy, bf.shape[0]


def reassemble_stack(stack_path, pred_dirs, index, n_slices, foreground_label_values, stack_src_root, stack_out_root):
    """Rebuild ZCYX (ch0=brightfield, ch1..=one label channel per entry in pred_dirs/foreground_label_values)
    for one stack from its per-slice predictions. pred_dirs and foreground_label_values are parallel lists,
    one per model/task. Returns (output_path, label_channels) where label_channels is a list of (Z,Y,X) arrays."""
    bf = read_timelapse_bf(stack_path)
    label_channels = []
    for pred_dir, foreground_label_value in zip(pred_dirs, foreground_label_values):
        labels = np.zeros(bf.shape, dtype=np.uint16)
        for z in range(n_slices):
            pred = sitk.GetArrayFromImage(sitk.ReadImage(str(pred_dir / f'{index:03d}_{z:04d}.nii.gz')))
            labels[z] = (pred[0] == foreground_label_value).astype(np.uint16)
        label_channels.append(labels)
    stack = np.stack([bf.astype(np.uint16), *label_channels], axis=1)
    rel = stack_path.relative_to(stack_src_root)
    out = stack_out_root / rel.with_name(rel.stem + '_labels.tif')
    out.parent.mkdir(parents=True, exist_ok=True)
    tifffile.imwrite(str(out), stack, imagej=True, metadata={'axes': 'ZCYX'})
    return out, label_channels

In [7]:
def segment_folder(tifs, pred_in, pred_out, pred_tif_dir, dataset_id, config, fold, trainer, checkpoint,
                    foreground_label_value):
    """Predict a folder of new brightfield *.tiff frames and write a binary foreground mask *.tif per image.
    Raises if any image is missing pixel-size metadata -> fix/remove it and rerun."""
    tifs = list(tifs)
    bad_spacing = check_pixel_metadata(tifs)
    if bad_spacing:
        raise ValueError(f'{len(bad_spacing)} image(s) missing pixel-size metadata -> remove or fix them, then rerun')
    for d in (pred_in, pred_out, pred_tif_dir):
        d.mkdir(parents=True, exist_ok=True)
    meta = {}
    for k, tif in enumerate(tifs):
        bf = np.asarray(tifffile.imread(str(tif))).astype(np.float32)
        case = f'pred_{k:03d}'
        spacing_xy = get_spacing_xy(tif)
        write_nii_2d(bf, pred_in / f'{case}_0000.nii.gz', is_label=False, spacing_xy=spacing_xy)
        meta[case] = tif
    run(f'nnUNetv2_predict -i {pred_in} -o {pred_out} -d {dataset_id} -c {config} '
        f'-f {fold} -tr {trainer} -chk {checkpoint}')
    for case, tif in meta.items():
        pred = sitk.GetArrayFromImage(sitk.ReadImage(str(pred_out / f'{case}.nii.gz')))
        mask = (pred[0] == foreground_label_value).astype(np.uint8)   # drop singleton z
        out = pred_tif_dir / f'{tif.stem}_mask.tif'
        tifffile.imwrite(str(out), mask)
        print('wrote', out.name, '| foreground%=', round(100 * float(mask.mean()), 2))
    print('masks ->', pred_tif_dir)

## Configure the two tasks (organoid & rosette)

In [8]:
nnunet_base = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\nnUNet")  # alongside the image/label pairs

tasks = [
    {
        'name': 'organoid',
        'dataset_id': 1,
        'dataset_name': 'Dataset001_Organoid',
        'images_dir': Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\organoid\tiff"), 
        'labels_dir': Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\organoid\labels"),            
        'image_glob': '*.tiff',
        'label_ext': '.tif',                        # labels share the image stem but use .tif
        'channel_names': {0: 'brightfield'},
        'labels': {'background': 0, 'organoid': 1},
        'foreground_label_value': 1,
        'config': '2d',
        'fold': 'all',                              # train on all cases; use a 0-4 fold for a CV Dice curve
        'trainer': 'nnUNetTrainer_250epochs_preview',  # 250-epoch trainer that also saves preview PNGs
        'checkpoint': 'checkpoint_best.pth',
    },
    {
        'name': 'rosette',
        'dataset_id': 2,
        'dataset_name': 'Dataset002_Rosette',
        'images_dir': Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\rosette\tiff\completed"),
        'labels_dir': Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\rosette\labels\completed"),
        'image_glob': '*.tiff',
        'label_ext': '.tif',
        'channel_names': {0: 'brightfield'},
        'labels': {'background': 0, 'rosette': 1},
        'foreground_label_value': 1,
        'config': '2d',
        'fold': 'all',
        'trainer': 'nnUNetTrainer_250epochs_preview',
        'checkpoint': 'checkpoint_best.pth',
    },
]

nnunet_dirs = setup_nnunet_dirs(nnunet_base)
for task in tasks:
    task['raw_ds'] = nnunet_dirs['raw'] / task['dataset_name']
    print(f"{task['name']:8s} images: {task['images_dir']}")
    print(f"{task['name']:8s} labels: {task['labels_dir']}")
    print(f"{task['name']:8s} raw ds: {task['raw_ds']}")

organoid images: Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\organoid\tiff
organoid labels: Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\organoid\labels
organoid raw ds: Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\nnUNet\nnUNet_raw\Dataset001_Organoid
rosette  images: Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\rosette\tiff\completed
rosette  labels: Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\rosette\labels\completed
rosette  raw ds: Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\nnUNet\nnUNet_raw\Dataset002_Rosette


## Convert both tasks to nnU-Net raw format

In [ ]:
for task in tasks:
    images_dir, labels_dir = task['images_dir'], task['labels_dir']
    image_glob, label_ext, raw_ds = task['image_glob'], task['label_ext'], task['raw_ds']
    dataset_name = task['dataset_name']

    img_paths = sorted(images_dir.glob(image_glob))
    missing_labels = [p.name for p in img_paths if not (labels_dir / (p.stem + label_ext)).exists()]
    print(f'{dataset_name}: {len(img_paths)} images found; {len(missing_labels)} without a matching label',
          missing_labels[:5])

    bad_spacing = check_pixel_metadata(img_paths)
    if bad_spacing:
        raise ValueError(f'{dataset_name}: {len(bad_spacing)} image(s) missing pixel-size metadata -> '
                          'remove or fix them, then rerun')

    build_dataset(img_paths, labels_dir, label_ext, raw_ds)
    n_train = len(list((raw_ds / 'labelsTr').glob('*.nii.gz')))
    generate_dataset_json(str(raw_ds), channel_names=task['channel_names'], labels=task['labels'],
                          num_training_cases=n_train, file_ending='.nii.gz', dataset_name=dataset_name)
    task['n_train'] = n_train
    print(f'{dataset_name}: dataset.json written with {n_train} training cases')

## Plan, preprocess (both tasks)

In [ ]:
for task in tasks:
    run(f"nnUNetv2_plan_and_preprocess -d {task['dataset_id']} --verify_dataset_integrity")
    plans = nnunet_dirs['preprocessed'] / task['dataset_name'] / 'nnUNetPlans.json'
    if plans.exists():
        cfg = json.loads(plans.read_text())['configurations'].get(task['config'], {})
        print(f"{task['name']}: target spacing={cfg.get('spacing')}  "
              f"patch={cfg.get('patch_size')}  batch={cfg.get('batch_size')}")

## Train (run one task at a time)

Set `task` to `tasks[0]` (organoid) or `tasks[1]` (rosette), then run the cells below. Repeat for the other task once this one finishes.

In [ ]:
task = tasks[0]   

preview_dir = task['images_dir'].parent / 'training_previews'
preview_every = 10                       # save one preview every N epochs
preview_dir.mkdir(parents=True, exist_ok=True)
os.environ['NNUNET_PREVIEW_DIR']   = str(preview_dir)
os.environ['NNUNET_PREVIEW_EVERY'] = str(preview_every)

# Long-running. da_workers=0 avoids the silently-dying augmentation workers on Windows.
# You can also run this exact command in a `vascumap_nnunet` terminal instead.
run(f"nnUNetv2_train {task['dataset_id']} {task['config']} {task['fold']} -tr {task['trainer']}", da_workers=0)

In [ ]:
import glob
folder = r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\organoid\labels"



organoid_tifs = glob.glob(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\organoid\labels\*.tif")
print(f"Found {len(organoid_tifs)} tif files")
print(len(organoid_tifs))

In [ ]:
from IPython.display import Image, display
run_dir = nnunet_dirs['results'] / task['dataset_name'] / f"{task['trainer']}__nnUNetPlans__{task['config']}" / f"fold_{task['fold']}"
prog = run_dir / 'progress.png'
display(Image(str(prog))) if prog.exists() else print('no progress.png yet:', run_dir)

# Latest input / ground-truth / prediction preview saved next to the data.
previews = sorted(preview_dir.glob('preview_epoch_*.png'))
display(Image(str(previews[-1]))) if previews else print('no preview PNG yet in', preview_dir)

In [ ]:
tasks[1]["images_dir"]

In [ ]:
task = tasks[1]   # EDIT: switch to tasks[1] to train the rosette model instead

preview_dir = task['images_dir'].parent / 'training_previews'
preview_every = 10                       # save one preview every N epochs
preview_dir.mkdir(parents=True, exist_ok=True)
os.environ['NNUNET_PREVIEW_DIR']   = str(preview_dir)
os.environ['NNUNET_PREVIEW_EVERY'] = str(preview_every)

# Long-running. da_workers=0 avoids the silently-dying augmentation workers on Windows.
# You can also run this exact command in a `vascumap_nnunet` terminal instead.
run(f"nnUNetv2_train {task['dataset_id']} {task['config']} {task['fold']} -tr {task['trainer']}", da_workers=0)

In [ ]:

run_dir = nnunet_dirs['results'] / task['dataset_name'] / f"{task['trainer']}__nnUNetPlans__{task['config']}" / f"fold_{task['fold']}"
prog = run_dir / 'progress.png'
display(Image(str(prog))) if prog.exists() else print('no progress.png yet:', run_dir)

# Latest input / ground-truth / prediction preview saved next to the data.
previews = sorted(preview_dir.glob('preview_epoch_*.png'))
display(Image(str(previews[-1]))) if previews else print('no preview PNG yet in', preview_dir)

## Predict on a folder of new brightfield images

Point `predict_src` at a folder of new `*.tiff` brightfield frames for the currently-selected `task`. Each is staged as a `(1,Y,X)` NIfTI using its own header pixel size, predicted, and written back out as a binary foreground mask `*.tif`.

In [ ]:
task = tasks[1]
predict_src = Path(r'Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\rosette\tiff\to_do')                     # EDIT: folder of new brightfield *.tiff to segment
pred_in  = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\rosette\predictions_on_unseen_tiffs\converted_inputs")  
pred_out = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\rosette\predictions_on_unseen_tiffs\converted_outputs")
pred_tif = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\rosette\predictions_on_unseen_tiffs\output_labels")
predict_tifs = sorted(predict_src.glob('*.tiff'))
print(f'{len(predict_tifs)} images to segment')
segment_folder(predict_tifs, pred_in, pred_out, pred_tif, task['dataset_id'], task['config'], task['fold'],
                task['trainer'], task['checkpoint'], task['foreground_label_value'])

## Predict on time-lapse stacks

Segments every slice of each stack in both `stack_srcs` folders with **both** models (organoid & rosette) independently, then saves a 3-channel (ch0=brightfield, ch1=organoid label, ch2=rosette label) stack to `stack_out`. Pixel-size metadata is missing on these stacks, so `PIXEL_SIZE_OVERRIDE_UM` is applied instead of raising.

In [9]:
import shutil

PIXEL_SIZE_OVERRIDE_CM = 0.00645   # metadata lost for these stacks: 1 px = 0.0064500 cm = 64.5 um

stack_srcs = [
    Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\Stitched_videos"),
    Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\NXD11\Stitched_videos"),
]

stack_in_nii = nnunet_base / 'stack_in'
stack_pred_dirs = {t['name']: nnunet_base / f"stack_pred_{t['name']}" for t in tasks}
pred_dirs  = [stack_pred_dirs[t['name']] for t in tasks]
fg_values  = [t['foreground_label_value'] for t in tasks]

for stack_src in stack_srcs:
    # Staged NIfTI slices + per-model predictions live alongside the source stacks (not in the shared
    # nnunet_base), so they're easy to find/clean up per-experiment and NXD8/NXD11 never collide.
    intermediates_dir = stack_src.parent / 'nnunet_stack_intermediates'
    stack_in_nii = intermediates_dir / 'stack_in'
    stack_pred_dirs = {t['name']: intermediates_dir / f"stack_pred_{t['name']}" for t in tasks}
    pred_dirs = [stack_pred_dirs[t['name']] for t in tasks]

    # Clear staging/prediction dirs per source folder so no stale slices from a previous run linger.
    for d in (stack_in_nii, *stack_pred_dirs.values()):
        shutil.rmtree(d, ignore_errors=True)
        d.mkdir(parents=True, exist_ok=True)

    stack_tifs = sorted(stack_src.rglob('*.tif*'))
    check_pixel_metadata(stack_tifs, override_cm=PIXEL_SIZE_OVERRIDE_CM)   # override applied; nothing to fix

    # Stage every slice of every stack once (the staged input is the same regardless of which model predicts it).
    counts = {}
    for i, p in enumerate(stack_tifs):
        spacing_xy, counts[i] = stage_stack(p, stack_in_nii, i, override_cm=PIXEL_SIZE_OVERRIDE_CM)
        print(f'  {p.name}  spacing_xy={spacing_xy}  slices={counts[i]}')
    print(f'{stack_src}: {len(stack_tifs)} stacks, {sum(counts.values())} slices staged; predicting...')

    # Predict with both models independently.
    # --disable_tta skips mirrored test-time-augmentation passes (~2-4x fewer forward passes per tile).
    # -step_size 0.75 reduces sliding-window tile overlap from the 0.5 default (fewer tiles per case).
    # Together these should cut runtime substantially with only a small effect on mask quality/smoothness.
    for t in tasks:
        run(f"nnUNetv2_predict -i {stack_in_nii} -o {stack_pred_dirs[t['name']]} -d {t['dataset_id']} -c {t['config']} "
            f"-f {t['fold']} -tr {t['trainer']} -chk {t['checkpoint']} --disable_tta -step_size 0.75")

    # Reassemble per stack: ZCYX with ch0=brightfield, ch1=organoid label, ch2=rosette label; mirror sub-folders.
    # out_root lives beside stack_src (e.g. NXD8/combined_stacks), not inside it, so it's never picked up by the rglob above.
    out_root = stack_src.parent / "combined_stacks"
    out_root.mkdir(parents=True, exist_ok=True)

    for i, p in enumerate(stack_tifs):
        out, label_channels = reassemble_stack(p, pred_dirs, i, counts[i], fg_values, stack_src, out_root)
        fg_pcts = ', '.join(f'{t["name"]}={100 * float(l.mean()):.2f}%' for t, l in zip(tasks, label_channels))
        print(f'{p.relative_to(stack_src).as_posix()}  ({counts[i]} slices)  {fg_pcts}  -> {out}')


24/24 images missing pixel-size metadata -> using override_cm=0.00645 for these
Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\Stitched_videos\A1_Neurons_NXD8.tif
Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\Stitched_videos\A2_NCO013T_NXD8.tif
Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\Stitched_videos\A3_NCO015T_NXD8.tif
Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\Stitched_videos\A4_mLCNEC11_NXD8.tif
Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\Stitched_videos\A5_mLCNEC23_NXD8.tif
Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\Stitched_videos\A6_mLCNEC26_NXD8.tif
Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\Stitched_videos\B1_Neurons_NXD8.tif
Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\Stitched_videos\B2_NCO013T_NXD8.tif
Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\Stitched_videos\B3_NCO015T_NXD8.tif
Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\Stitched_videos\B4_mLCNEC11_NXD8.tif
Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\Stitched_videos\B

c:\Users\taylorhearn\AppData\Local\miniconda3\envs\vascumap_nnunet\Lib\site-packages\tifffile\tifffile.py:3809: UserWarning: <tifffile.TiffWriter 'A1_Neurons_NXD8_labels.tif'> truncating ImageJ file
  self._write_remaining_pages()


A1_Neurons_NXD8.tif  (16 slices)  organoid=0.04%, rosette=1.11%  -> Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\combined_stacks\A1_Neurons_NXD8_labels.tif


c:\Users\taylorhearn\AppData\Local\miniconda3\envs\vascumap_nnunet\Lib\site-packages\tifffile\tifffile.py:3809: UserWarning: <tifffile.TiffWriter 'A2_NCO013T_NXD8_labels.tif'> truncating ImageJ file
  self._write_remaining_pages()


A2_NCO013T_NXD8.tif  (16 slices)  organoid=0.04%, rosette=2.19%  -> Y:\Patryk_Polinski\Bel_Marta_Classification\NXD8\combined_stacks\A2_NCO013T_NXD8_labels.tif


...

D4_mLCNEC23_NXD11.tif  (13 slices)  organoid=1.44%, rosette=4.01%  -> Y:\Patryk_Polinski\Bel_Marta_Classification\NXD11\combined_stacks\D4_mLCNEC23_NXD11_labels.tif


c:\Users\taylorhearn\AppData\Local\miniconda3\envs\vascumap_nnunet\Lib\site-packages\tifffile\tifffile.py:3809: UserWarning: <tifffile.TiffWriter 'D5_mLCNEC26_NXD11_labels.tif'> truncating ImageJ file
  self._write_remaining_pages()


D5_mLCNEC26_NXD11.tif  (13 slices)  organoid=4.63%, rosette=2.71%  -> Y:\Patryk_Polinski\Bel_Marta_Classification\NXD11\combined_stacks\D5_mLCNEC26_NXD11_labels.tif


c:\Users\taylorhearn\AppData\Local\miniconda3\envs\vascumap_nnunet\Lib\site-packages\tifffile\tifffile.py:3809: UserWarning: <tifffile.TiffWriter 'D6_mLCNEC27_NXD11_labels.tif'> truncating ImageJ file
  self._write_remaining_pages()


D6_mLCNEC27_NXD11.tif  (13 slices)  organoid=1.62%, rosette=1.92%  -> Y:\Patryk_Polinski\Bel_Marta_Classification\NXD11\combined_stacks\D6_mLCNEC27_NXD11_labels.tif


In [ ]:
import glob
output_label_folder = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\rosette\labels\completed")
output_labels = glob.glob(str(output_label_folder / "*.tif"))
output_labels.sort()
unseen_tif_folder = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\rosette\tiff\completed")
unseen_tifs = glob.glob(str(unseen_tif_folder / "*.tiff"))
unseen_tifs.sort()

print(len(unseen_tifs), len(output_labels))

for i in range(len(unseen_tifs)):
    unseen_image_name = Path(unseen_tifs[i]).stem
    unseen_tif = tifffile.imread(unseen_tifs[i])
    output_label = tifffile.imread(output_labels[i])
    fig, ax = plt.subplots(ncols = 2)
    ax[0].imshow(unseen_tif, cmap='gray')
    ax[0].set_title(f"Unseen Image: {unseen_image_name}")
    ax[1].imshow(output_label, cmap='gray')
    ax[1].set_title("Output Label")
    for a in ax:
        a.axis('off')
    plt.savefig(rf"y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\rosette\labels\visualisation_of_completed\{unseen_image_name}.png")